In [6]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt



# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name
username = "aacuser"
password = "Haitibyentomkwesa"
shelter = AnimalShelter(username, password)

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read()).decode()

app.layout = html.Div([
    # Header Section with Logo and Student Identifier
    html.Div(className='row', style={'display': 'flex', 'alignItems': 'center', 'padding': '10px'}, children=[
        html.A([
            html.Img(src=f'data:image/png;base64,{encoded_image}', style={'height': '100px', 'padding-right': '20px'})
        ], href='https://www.snhu.edu'),
        html.Div([
            html.H1('Grazioso Salvare Search and Rescue Dashboard', style={'margin': '0'}),
            html.H3('Developed by: [Elijah Bastien] - [6/20/26]', style={'margin': '5px 0 0 0', 'color': 'gray'})
        ])
    ]),
    html.Hr(),
    
    # Interactive Filtering Options (Radio Buttons)
    html.Div([
        html.Label(html.B('Select Rescue Filter Type:')),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'Water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'Mountain'},
                {'label': 'Disaster or Individual Tracking', 'value': 'Disaster'},
                {'label': 'Reset (All Records)', 'value': 'Reset'}
            ],
            value='Reset',
            labelStyle={'display': 'inline-block', 'margin-right': '20px', 'margin-top': '10px'}
        )
    ], style={'padding': '10px'}),
    html.Hr(),
    
    # Interactive Data Table
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        page_size=10,
        sort_action="native",
        filter_action="native",
        row_selectable="single",
        selected_rows=[0],
        style_table={'overflowX': 'auto'}
    ),
    html.Br(),
    html.Hr(),
    
    # Layout for charts side-by-side
    html.Div(className='row', style={'display': 'flex'}, children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',
            style={'width': '50%', 'padding': '10px'}
        ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            style={'width': '50%', 'padding': '10px'}
        )
    ])
])

#############################################
# Interaction Between Components / Controller
#############################################

# Callback to filter the data table based on MongoDB queries
@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):
    if filter_type == 'Water':
        query = {
            'animal_type': 'Dog',
            'breed': {'$in': ['Labrador Retriever Mix', 'Chesapeake Bay Retriever', 'Newfoundland']},
            'sex_upon_outcome': 'Spayed Female',
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}
        }
    elif filter_type == 'Mountain':
        query = {
            'animal_type': 'Dog',
            'breed': {'$in': ['German Shepherd', 'Alaskan Malamute', 'Siberian Husky', 'Rottweiler', 'St. Bernard']},
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}
        }
    elif filter_type == 'Disaster':
        query = {
            'animal_type': 'Dog',
            'breed': {'$in': ['Doberman Pinscher', 'Golden Retriever', 'Bloodhound', 'Rottweiler']},
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {'$gte': 20, '$lte': 300}
        }
    else:  # Reset / Default state
        query = {}

    # Query database and convert to clean dictionary format
    filtered_df = pd.DataFrame.from_records(db.read(query))
    if '_id' in filtered_df.columns:
        filtered_df.drop(columns=['_id'], inplace=True)
        
    return filtered_df.to_dict('records')


# Callback to update dynamic Pie Chart based on current data table rows
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):
    if viewData is None or len(viewData) == 0:
        return html.Div("No data available to display chart.")
    
    dff = pd.DataFrame.from_dict(viewData)
    
    # Generate dynamic pie chart showing breed distribution
    fig = px.pie(
        dff, 
        names='breed', 
        title='Breed Breakdown of Screened Candidates',
        hole=.3
    )
    
    return [dcc.Graph(figure=fig)]


# Highlight selected column/cells
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if selected_columns is None:
        return []
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# Callback to update geo-location map
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, index):  
    if viewData is None or len(viewData) == 0:
        return html.Div("No spatial data available.")
        
    # Handle empty list indexing safe-checks
    if index is None or len(index) == 0:
        row = 0
    else: 
        row = index[0]
        
    dff = pd.DataFrame.from_dict(viewData)
    
    # Column verification safeguards (adjust numbers if dataset maps differently)
    try:
        lat = dff.iloc[row]['location_lat']
        lon = dff.iloc[row]['location_long']
        breed = dff.iloc[row]['breed']
        name = dff.iloc[row]['name']
    except (KeyError, IndexError):
        # Fallback to column index rules if names differ
        lat = dff.iloc[row, 13]
        lon = dff.iloc[row, 14]
        breed = dff.iloc[row, 4]
        name = dff.iloc[row, 9]
        
    return [
        dl.Map(style={'width': '100%', 'height': '500px'}, center=[lat, lon], zoom=12, children=[
            dl.TileLayer(id="base-layer-id"),
            dl.Marker(position=[lat, lon], children=[
                dl.Tooltip(f"Breed: {breed}"),
                dl.Popup([
                    html.H4(f"Animal Name: {name if name else 'Unnamed'}"),
                    html.P(f"Breed: {breed}")
                ])
            ])
        ])
    ]

# Run app server
if __name__ == '__main__':
    app.run_server(debug=True)

Dash app running on https://concertslogan-simplepermit-3000.codio.io/proxy/8050/
